# KRISIS v0 — Foundation

**Your Smart Triage System.** This notebook works through *every* aspect of the **v0 / Day 1** scope from the project plan, end to end:

1. Domain story finalized
2. Ticket format & output JSON schema defined
3. Category + team taxonomy defined
4. Priority rules defined
5. Few-shot prompt drafted with the three required edge cases (angry tone, vague/short, ambiguous)
6. One raw LLM API call made "by hand" to confirm the mechanism works end to end

> **Scope note:** v0 is about *locking the design* and *proving the mechanism once*. No FastAPI, no Streamlit, no database yet — those arrive in v1. The only cell that needs network access or an API key (and costs money) is the "raw API call" in section 6.

## 0. Setup

Recommended: run this notebook inside a virtual environment so you don't install into system Python.

```bash
python3 -m venv .venv
source .venv/bin/activate
pip install openai pydantic python-dotenv ipykernel
python -m ipykernel install --user --name krisis-v0
```

Then select the **krisis-v0** kernel. Put your key in a `.env` file next to this notebook:

```
OPENAI_API_KEY=sk-...
```

In [ ]:
# Run once if the packages aren't already in this kernel.
%pip install --quiet openai pydantic python-dotenv

In [ ]:
import os
import json
from typing import Literal

from pydantic import BaseModel, Field
from dotenv import load_dotenv

load_dotenv()  # loads OPENAI_API_KEY from a .env file if present
print("OPENAI_API_KEY set:", bool(os.getenv("OPENAI_API_KEY")))

## 1. Domain story

KRISIS is a smart triage system for the internal IT and engineering support desk of an AI-first software engineering company. Employees submit free-text tickets covering access requests, infrastructure issues, CI/CD failures, security concerns, developer-tooling problems, and hardware issues.

Today a human reads every ticket and manually decides its **category**, **priority**, and **owning team** before any work starts. That first sorting step is repetitive, slow, and inconsistent between people. KRISIS automates it: given a raw ticket message, it returns a structured decision so the ticket is already sorted by the time a human looks at it.

## 2. Ticket format & output JSON schema

**Input:** a single raw text message, exactly as an employee would type it. Nothing else is required.

**Final output (the contract):** a JSON object with four fields — `category`, `priority`, `assigned_team`, `reasoning`.

The design splits **judgement** (the model's job) from **mapping** (deterministic code):

- **`TicketDecision` — the LLM output schema.** The model reasons *first* (`analysis`), then commits to `impact`, `urgency`, `category`, and a one-line `reasoning`. Field order matters: because structured outputs are generated in order, putting the assessment before the decision forces the model to actually think rather than rationalize after the fact.
- **`priority` is derived in code** from `impact × urgency` via a fixed matrix (+ overrides) — see section 4.
- **`assigned_team` is derived in code** from `category` (`TAXONOMY[category]["team"]`).

Why: routing and prioritising are *lookups*, not judgement calls. Deriving them removes whole failure modes (team can't mismatch the category; priority can't drift for the same assessment), makes those fields 100% consistent across identical runs, and keeps one source of truth. The model's `impact`/`urgency`/`analysis` are surfaced alongside the output for transparency (so a priority is *defensible*), but they aren't part of the response contract. The LLM output schema also doubles as the structured-output schema in section 6 (and, later, the FastAPI response model built the same way).

In [ ]:
Category = Literal[
    "access_iam", "infra_outage", "ci_cd", "security",
    "dev_tooling", "hardware", "unclassified",
]
Priority = Literal["High", "Medium", "Low"]  # the DERIVED value (not chosen by the LLM)

# The model's two assessment axes (see section 4).
Impact  = Literal["broad", "narrow"]
Urgency = Literal["blocked", "workaround"]


class TicketDecision(BaseModel):
    """What the LLM returns. Field ORDER matters: the model reasons first, then commits.
    priority and assigned_team are NOT here - they are derived in code from these fields."""
    analysis: str = Field(..., description="Think first: weigh impact, urgency, and category")
    impact:  Impact  = Field(..., description="broad = many people / shared or critical service; narrow = one person / small scope")
    urgency: Urgency = Field(..., description="blocked = no usable workaround; workaround = work can continue")
    category: Category = Field(..., description="One of the fixed taxonomy categories")
    reasoning: str = Field(..., description="One line citing the impact + urgency that drove the decision")


print(json.dumps(TicketDecision.model_json_schema(), indent=2))

## 3. Domain taxonomy

Seven categories, each with a fixed owning team. Kept as data so the prompt, validation, and (later) the API all read from one source of truth.

In [ ]:
TAXONOMY = {
    "access_iam":   {"team": "Identity and access",       "description": "Login, permissions, SSO, credential issues"},
    "infra_outage": {"team": "Infrastructure operations", "description": "Server, network, or cloud infrastructure failures"},
    "ci_cd":        {"team": "Platform engineering",      "description": "Build and deployment pipeline failures"},
    "security":     {"team": "Security team",             "description": "Vulnerabilities, suspicious activity, security incidents"},
    "dev_tooling":  {"team": "Developer experience",      "description": "IDE, internal tools, environment setup issues"},
    "hardware":     {"team": "Hardware support",          "description": "Laptop, peripheral, or device issues"},
    "unclassified": {"team": "Triage",                    "description": "Ticket lacks enough detail to classify confidently"},
}

for cat, info in TAXONOMY.items():
    print(f"{cat:14s} -> {info['team']}")

## 4. Priority rules

Priority is **computed, not guessed**. The model assesses two axes — and priority falls out of a fixed matrix. This is based on the standard IT service-management model (**Priority = Impact × Urgency**), and it deliberately excludes tone: an angry ticket is never more urgent than a calm one describing the same problem.

- **Impact** — `broad` (many people, a shared/customer-facing service, or core infra) vs. `narrow` (one person / small scope).
- **Urgency** — `blocked` (no usable workaround, or an imminent externally-imposed deadline) vs. `workaround` (work can continue, or it's non-blocking/cosmetic/a request). *("Deadline" is folded into urgency rather than being its own fuzzy trigger.)*

**Matrix:**

| | blocked | workaround |
|---|---|---|
| **broad** | High | Medium |
| **narrow** | Medium | Low |

**Overrides (applied in code, after the matrix):**
- `category == "security"` → **High** (a security incident escalates regardless of scope).
- `category == "unclassified"` → **Medium** (safe default, flagged for review).

This resolves the old rules' problems: no collisions (every `impact × urgency` maps to exactly one tier), no subjective triggers (the axes are defined), and it's defensible — you can show the full `impact → urgency → matrix → priority` chain for any ticket.

In [ ]:
# Priority is not a free judgement - it is a lookup on two assessed axes.

IMPACT_LEVELS = {
    "broad":  "Multiple people, a whole team, a shared/customer-facing service, or core infra (auth, network, CI that blocks all merges)",
    "narrow": "A single person or a small, non-critical scope",
}
URGENCY_LEVELS = {
    "blocked":    "Cannot do core work and no usable workaround (or an imminent, externally-imposed deadline with no workaround)",
    "workaround": "Work can continue - a workaround exists, or the issue is non-blocking / cosmetic / a request or question",
}

# Impact x Urgency -> Priority
PRIORITY_MATRIX = {
    ("broad",  "blocked"):    "High",
    ("broad",  "workaround"): "Medium",
    ("narrow", "blocked"):    "Medium",
    ("narrow", "workaround"): "Low",
}

# Overrides applied in code AFTER the matrix (see derive_priority in section 6).
OVERRIDES = {
    "security":     "High   - a security incident escalates regardless of assessed scope",
    "unclassified": "Medium - safe default, flagged for human review",
}

print("Impact x Urgency -> Priority")
print(f"{'':10s} {'blocked':>10s} {'workaround':>12s}")
for imp in ("broad", "narrow"):
    print(f"{imp:10s} {PRIORITY_MATRIX[(imp,'blocked')]:>10s} {PRIORITY_MATRIX[(imp,'workaround')]:>12s}")
print("\nOverrides:")
for cat, rule in OVERRIDES.items():
    print(f"  {cat:14s} -> {rule}")

## 5. Few-shot prompt (with the three required edge cases)

The prompt is few-shot: a system message that states the role, taxonomy, and rules, followed by worked examples. The three required edge cases are covered, and every example's `reasoning` names the rule that drove the decision, so the logic is checkable:

1. **Angry tone** — must not inflate priority.
2. **Very short / vague** — falls back to `unclassified` + Triage.
3. **Ambiguous** — fits more than one category; the reasoning states the tie-break.

In [ ]:
def build_system_prompt() -> str:
    taxonomy_lines = "\n".join(
        f"- {cat}: {info['description']} (routed to: {info['team']})"
        for cat, info in TAXONOMY.items()
    )
    impact_lines  = "\n".join(f"- {k}: {v}" for k, v in IMPACT_LEVELS.items())
    urgency_lines = "\n".join(f"- {k}: {v}" for k, v in URGENCY_LEVELS.items())
    return f"""You are KRISIS, a smart triage system for an internal IT and engineering support desk.

For each raw ticket, reason step by step, THEN commit:
1. analysis: briefly weigh the situation - what is affected, who is blocked, is there a workaround.
2. impact: choose one.
3. urgency: choose one.
4. category: choose exactly one.
5. reasoning: one line naming the impact + urgency (and any tie-break) that drove the decision.

You do NOT choose priority or the team. Priority is computed from impact x urgency, and the
team is assigned from the category - both automatically, downstream. Judge on facts, not tone:
an angry ticket is not more urgent than a calm one describing the same problem.

Impact:
{impact_lines}

Urgency:
{urgency_lines}

Categories (pick exactly one) and the team each is routed to:
{taxonomy_lines}

Calibration anchors (impact + urgency -> resulting priority):
- broad + blocked  -> High   e.g. "Prod API returns 500s for everyone, no workaround."
- broad + workaround -> Medium  e.g. "Team wiki is down but the info is mirrored in Slack."
- narrow + blocked -> Medium  e.g. "I'm locked out of my account and can't work."
- narrow + workaround -> Low  e.g. "Typo on the internal dashboard header." / "Please grant me access to the analytics tool."

Rules:
- If the message lacks enough detail to classify confidently, use category "unclassified" and
  neutral defaults impact="narrow", urgency="workaround" (priority is set to Medium downstream).
- A security incident is category "security" regardless of scope.
"""


print(build_system_prompt())

In [ ]:
FEWSHOT = [
    # Edge case 1 - angry tone must NOT inflate priority
    {
        "ticket": "This is RIDICULOUS. I've been locked out of my account for two hours and nobody is helping. Reset my password NOW.",
        "decision": {
            "analysis": "One person locked out of their account, no workaround mentioned. Angry tone is irrelevant to impact. This is an account/credential issue.",
            "impact": "narrow",
            "urgency": "blocked",
            "category": "access_iam",
            "reasoning": "narrow + blocked -> Medium; category access_iam (account lockout). Tone ignored per the impact-only rule.",
        },
    },
    # Edge case 2 - very short / vague message
    {
        "ticket": "it's broken",
        "decision": {
            "analysis": "No system, scope, or impact can be identified from the message. Not enough detail to classify confidently.",
            "impact": "narrow",
            "urgency": "workaround",
            "category": "unclassified",
            "reasoning": "Insufficient-detail rule -> unclassified with neutral defaults; priority set to Medium downstream, flagged for review.",
        },
    },
    # Edge case 3 - could fit more than one category
    {
        "ticket": "I can't push my code - the CI pipeline rejects my SSH key as unauthorized.",
        "decision": {
            "analysis": "Symptom is in CI, but the root cause is an unauthorized credential (SSH key), which is access/IAM. Affects one engineer, who is blocked from pushing.",
            "impact": "narrow",
            "urgency": "blocked",
            "category": "access_iam",
            "reasoning": "Fits ci_cd and access_iam; chose access_iam because the root cause is an unauthorized credential, not a pipeline defect. narrow + blocked -> Medium.",
        },
    },
]
print(len(FEWSHOT), "few-shot examples")

In [ ]:
def build_messages(ticket: str):
    messages = [{"role": "system", "content": build_system_prompt()}]
    for ex in FEWSHOT:
        messages.append({"role": "user", "content": ex["ticket"]})
        messages.append({"role": "assistant", "content": json.dumps(ex["decision"])})
    messages.append({"role": "user", "content": ticket})
    return messages


# Preview the assembled conversation for a sample ticket
for m in build_messages("VPN keeps dropping every few minutes on my laptop."):
    print(m["role"].upper(), ":", m["content"][:110].replace("\n", " "))

## 6. One raw API call, by hand

This is the single cell that needs an `OPENAI_API_KEY` and network access. We send the few-shot prompt plus a real ticket; OpenAI **structured outputs** returns a `TicketDecision` where the model has reasoned first (`analysis` → `impact` → `urgency`) and then committed to a `category` and one-line `reasoning`.

Code then derives the rest: `priority` from the `impact × urgency` matrix (+ overrides) and `assigned_team` from the category. The final dict carries the canonical four fields plus the model's assessment for transparency.

**Usage readout:** after every call, `classify()` prints this call's token count and the **per-minute rate-limit** usage read from the response headers. That is the only "limit" a *project* API key can see — it's a per-minute allowance that refills each minute, **not** your monthly billing budget (that needs account/admin access, which you don't have here).

> If your `openai` SDK version doesn't have `client.beta.chat.completions.with_raw_response.parse`, use `client.chat.completions.with_raw_response.parse` (same arguments).

In [ ]:
from openai import OpenAI

MODEL = "gpt-4o-mini"  # cheap default; swap for "gpt-4o" for stronger reasoning
client = OpenAI()      # reads OPENAI_API_KEY from the environment


def derive_priority(d: TicketDecision) -> str:
    """Priority is computed, not chosen: overrides first, then the impact x urgency matrix."""
    if d.category == "security":
        return "High"      # a security incident escalates regardless of scope
    if d.category == "unclassified":
        return "Medium"    # safe default, flagged for review
    return PRIORITY_MATRIX[(d.impact, d.urgency)]


def _rate_limit_report(headers) -> str:
    """Per-minute rate-limit usage from response headers - the only 'limit' a project key can read.
    NOTE: this is a per-minute allowance that refills each minute, NOT a monthly billing budget."""
    def one(kind):  # kind in {"tokens", "requests"}
        lim = headers.get(f"x-ratelimit-limit-{kind}")
        rem = headers.get(f"x-ratelimit-remaining-{kind}")
        if lim is None or rem is None:
            return f"{kind}: n/a"
        lim_i, rem_i = int(lim), int(rem)
        used_pct = (lim_i - rem_i) / lim_i * 100 if lim_i else 0.0
        reset = headers.get(f"x-ratelimit-reset-{kind}", "?")
        return f"{kind} {used_pct:4.1f}% used ({rem_i}/{lim_i} left, resets in {reset})"
    return "  |  ".join(one(k) for k in ("tokens", "requests"))


def classify(ticket: str) -> dict:
    raw = client.beta.chat.completions.with_raw_response.parse(
        model=MODEL,
        messages=build_messages(ticket),
        response_format=TicketDecision,  # <- structured output; model reasons then commits
    )
    completion = raw.parse()
    d = completion.choices[0].message.parsed
    # After every run: this-call token count + per-minute rate-limit usage.
    u = completion.usage
    print(f"[usage] this call: {u.total_tokens} tok (in {u.prompt_tokens} / out {u.completion_tokens})")
    print(f"[rate-limit / minute] {_rate_limit_report(raw.headers)}")
    return {
        # --- canonical 4-field output contract ---
        "category": d.category,
        "priority": derive_priority(d),
        "assigned_team": TAXONOMY[d.category]["team"],
        "reasoning": d.reasoning,
        # --- the model's assessment, surfaced for transparency (not part of the contract) ---
        "impact": d.impact,
        "urgency": d.urgency,
        "analysis": d.analysis,
    }


result = classify("The whole prod database is down and no one can log in to the app.")
print(json.dumps(result, indent=2))

In [ ]:
# Fire the three required edge cases (plus one clear High) through the live model.
samples = [
    "This is RIDICULOUS, reset my password NOW, locked out for hours!",   # angry tone
    "it's broken",                                                         # vague
    "CI pipeline says my SSH key is unauthorized so I can't push",         # ambiguous
    "Prod API is returning 500s for everyone and there is no workaround.", # clear High
]
for s in samples:
    d = classify(s)
    print(f"\nTICKET: {s}")
    print(f"  think: impact={d['impact']} | urgency={d['urgency']}")
    print(f"  ->     {d['category']} | {d['priority']} | {d['assigned_team']}")
    print(f"         {d['reasoning']}")

## 7. What v0 confirms — and what's next

If section 6 runs and returns well-formed results with all four fields and sensible priorities (angry tone stays Medium, "it's broken" becomes unclassified, the SSH-key ticket disambiguates with a stated rule), then **v0 is done**: the domain, schema, taxonomy, priority rules, and prompt are locked, and the mechanism is proven end to end.

Note the design choice: the LLM decides `category` / `priority` / `reasoning`; `assigned_team` is derived in code, so routing is deterministic and can never mismatch the category.

**Next (v1):**
- Promote the final output to a dedicated `RoutedTicket` Pydantic model (all four fields, with `assigned_team` filled from `TAXONOMY`) — this becomes the **FastAPI** response model.
- Wrap the logic behind a **FastAPI** backend (`POST /classify`), add retry + fallback reliability, persist every request to **PostgreSQL**, and build the **Streamlit** UI as a client of the API.